# 0. Verificación de GPU

In [ ]:
import torch
print(f"PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}  |  GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  Dispositivo: {torch.cuda.get_device_name(0)}")

# 1. Configuración de rutas (EDITAR)

Edita las cuatro rutas según los datasets que tengas montados en Kaggle.
Las dos primeras (MODEL_DIR y CHECKPOINT) deberían apuntar al dataset con
el código del modelo y el checkpoint post fine-tuning. VOCAB_PATH apunta
al fichero `vocab.txt` que usaste durante el entrenamiento. TEST_SET_DIR
es el dataset nuevo que subes con los pares (.jpg, .txt) de prueba.

In [ ]:
from pathlib import Path

# ────────────────────────────────────────────────────────────────────────────
# Rutas — AJUSTAR según los datasets de Kaggle que tengas montados
# ────────────────────────────────────────────────────────────────────────────
MODEL_DIR    = Path("/kaggle/input/<dataset-modelo>/Model")          # carpeta con model.py, dataset.py, metrics.py
CHECKPOINT   = Path("/kaggle/input/<dataset-modelo>/checkpoints/best_model_finetune.pt")
VOCAB_PATH   = Path("/kaggle/input/<dataset-vocab>/vocab/vocab.txt")
TEST_SET_DIR = Path("/kaggle/input/test-set")                         # nuevo dataset con jpg+txt
# ────────────────────────────────────────────────────────────────────────────

# Comprobaciones tempranas
for label, p in [("MODEL_DIR", MODEL_DIR), ("CHECKPOINT", CHECKPOINT),
                 ("VOCAB_PATH", VOCAB_PATH), ("TEST_SET_DIR", TEST_SET_DIR)]:
    exists = "✓" if p.exists() else "✗ NO EXISTE"
    print(f"  {label:15} {exists}  {p}")

# 2. Configurar entorno e importar módulos del proyecto

In [ ]:
import os, sys, json
import numpy as np
from collections import defaultdict
from PIL import Image
from torchvision import transforms

# Variables de entorno que dataset.py lee al importarse
os.environ["OCR_VOCAB_PATH"] = str(VOCAB_PATH)

# Añadir el directorio del proyecto al path
sys.path.insert(0, str(MODEL_DIR))

from model   import CRNN
from dataset import (decode_ctc, autocrop_whitespace,
                     IMG_HEIGHT, CNN_STRIDE, encode, get_font_label)
import metrics

print(f"✓ Módulos importados desde {MODEL_DIR}")

# 3. Cargar modelo (checkpoint post fine-tuning)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(CHECKPOINT, map_location=device)
cfg  = ckpt.get("config", {})
model = CRNN(
    vocab_size  = cfg.get("vocab_size",  101),
    img_height  = cfg.get("img_height",  IMG_HEIGHT),
    hidden_size = cfg.get("hidden_size", 256),
    num_layers  = cfg.get("num_layers",  2),
)
# Descartar prefijo "_orig_mod." que añade torch.compile
state = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model_state"].items()}
model.load_state_dict(state)
model.to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"✓ Modelo cargado: {n_params:,} parámetros  |  dispositivo: {device}")
print(f"  Configuración del checkpoint:")
for k in ("vocab_size", "img_height", "hidden_size", "num_layers", "dropout"):
    if k in cfg: print(f"    {k:15} = {cfg[k]}")

# 4. Descubrir pares (imagen, transcripción) del test set

In [ ]:
img_exts = {".jpg", ".jpeg", ".png"}
test_pairs = []                   # (img_path, font, transcripción)
fonts_seen = defaultdict(int)
skipped    = {"sin_txt": 0, "txt_vacio": 0, "fuera_vocab": 0}

for p in sorted(Path(TEST_SET_DIR).rglob("*")):
    if p.suffix.lower() not in img_exts: continue
    txt = p.with_suffix(".txt")
    if not txt.exists():
        skipped["sin_txt"] += 1; continue
    text = txt.read_text(encoding="utf-8").strip()
    if not text:
        skipped["txt_vacio"] += 1; continue
    if len(encode(text)) == 0:
        skipped["fuera_vocab"] += 1; continue
    font = get_font_label(p)
    test_pairs.append((p, font, text))
    fonts_seen[font] += 1

print(f"✓ Test set cargado: {len(test_pairs)} pares válidos")
print(f"  Fuentes encontradas: {len(fonts_seen)}")
for font, cnt in sorted(fonts_seen.items()):
    print(f"    {font:30} {cnt:5} imágenes")
total_skip = sum(skipped.values())
if total_skip:
    print(f"  Descartados: sin_txt={skipped['sin_txt']}, "
          f"vacíos={skipped['txt_vacio']}, fuera_vocab={skipped['fuera_vocab']}")

# 5. Inferencia (decodificación greedy, por lotes)

La decodificación greedy es la misma que usa el protocolo de entrenamiento
para reportar todas las métricas de cap. 4. Si en otra ejecución quieres
medir el sistema en producción (beam + LM 5-grama), la celda opcional al
final del cuaderno lo cubre.

In [ ]:
from tqdm.auto import tqdm

IMG_H = cfg.get("img_height", 64)

tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

def preprocess(img_path):
    """Mismo preprocesado que NoAugSubset durante validación."""
    img = Image.open(img_path).convert("L")
    img = autocrop_whitespace(img, threshold=200, padding=2)
    w, h = img.size
    new_w = max(4, int(round(w * IMG_H / h)))
    img = img.resize((new_w, IMG_H), Image.BICUBIC)
    return tfm(img)   # tensor [1, IMG_H, new_w]


# Preprocesar todas las imágenes (en memoria)
print("Preprocesando imágenes...")
items = []
for p, font, text in tqdm(test_pairs):
    t = preprocess(p)
    items.append((t, font, text))

# Ordenar por ancho para agrupar tamaños similares y reducir padding
items.sort(key=lambda x: x[0].shape[2])


def collate_pad(batch_items):
    """Padding por la derecha al ancho máximo del lote."""
    tensors = [b[0] for b in batch_items]
    widths  = [t.shape[2] for t in tensors]
    max_w   = max(widths)
    h       = tensors[0].shape[1]
    batch   = torch.zeros(len(batch_items), 1, h, max_w)
    for i, t in enumerate(tensors):
        batch[i, :, :, :t.shape[2]] = t
    return batch, widths, [b[1] for b in batch_items], [b[2] for b in batch_items]


BATCH_SIZE = 32
hypotheses, references, fonts = [], [], []

print(f"\nEjecutando inferencia greedy (lotes de {BATCH_SIZE})...")
with torch.no_grad():
    for i in tqdm(range(0, len(items), BATCH_SIZE)):
        batch_items = items[i:i+BATCH_SIZE]
        batch, widths, bf, br = collate_pad(batch_items)
        batch = batch.to(device, non_blocking=True)

        log_probs = model(batch)              # [T, B, V]
        log_probs = log_probs.permute(1, 0, 2)  # [B, T, V]

        for j, (lp, w) in enumerate(zip(log_probs, widths)):
            valid_t  = w // CNN_STRIDE
            pred_idx = lp[:valid_t].argmax(dim=-1).cpu().tolist()
            hypotheses.append(decode_ctc(pred_idx))
            references.append(br[j])
            fonts.append(bf[j])

print(f"\n✓ Inferencia completada: {len(hypotheses)} predicciones")

# 6. Métricas globales sobre el test set

In [ ]:
print("\n" + "=" * 72)
print("RESULTADOS GLOBALES SOBRE TEST SET — DECODIFICACIÓN GREEDY")
print("=" * 72)

m_global = metrics.compute_all_metrics(hypotheses, references)

primary = [("CER", m_global["CER"]),
           ("WER", m_global["WER"]),
           ("Exactitud por línea", m_global["LineAcc"]),
           ("Char-F1", m_global["Char_F1"]),
           ("BLEU-4 (carácter)", m_global["BLEU4_char"]),
           ("1-NED", m_global["1-NED"])]

print(f"\n  {'Métrica':25} {'Valor':>10}")
print(f"  {'─'*25} {'─'*10}")
for name, val in primary:
    print(f"  {name:25} {val:10.4f}")

print(f"\n  N muestras totales: {m_global['N_samples']}")
print(f"  Coincidencias exactas: {m_global['Exact_matches']} de {m_global['N_samples']} "
      f"({100*m_global['Exact_matches']/m_global['N_samples']:.2f}%)")
print(f"  Longitud media (chars): ref={m_global['Avg_ref_len_char']:.1f}, "
      f"hyp={m_global['Avg_hyp_len_char']:.1f}")

# 7. Intervalos de confianza al 95% por bootstrap

In [ ]:
print("\nCalculando intervalos de confianza por bootstrap (10 000 remuestreos)...")

# Métricas por muestra para alimentar el bootstrap
cer_vals     = [metrics.cer(h, r) for h, r in zip(hypotheses, references)]
wer_vals     = [metrics.wer(h, r) for h, r in zip(hypotheses, references)]
line_correct = [1.0 if h == r else 0.0 for h, r in zip(hypotheses, references)]
char_f1_vals = [metrics._char_prf(h, r)[2] for h, r in zip(hypotheses, references)]

print(f"\n  {'Métrica':22} {'Media':>8} {'Std':>8}   {'IC 95%':>22}")
print(f"  {'─'*22} {'─'*8} {'─'*8}   {'─'*22}")
ci_summary = {}
for name, vals in [("CER", cer_vals),
                   ("WER", wer_vals),
                   ("Exactitud por línea", line_correct),
                   ("Char-F1", char_f1_vals)]:
    ci = metrics.bootstrap_ci(vals, n_bootstrap=10_000, seed=42)
    ci_summary[name] = ci
    mean = float(np.mean(vals))
    std  = float(np.std(vals, ddof=1))
    print(f"  {name:22} {mean:8.4f} {std:8.4f}   "
          f"[{ci['ci_low']:.4f}, {ci['ci_high']:.4f}]")

# 8. Desglose por fuente tipográfica + Kruskal-Wallis

In [ ]:
print("\n" + "=" * 72)
print("DESGLOSE POR FUENTE TIPOGRÁFICA")
print("=" * 72)

font_hyp = defaultdict(list)
font_ref = defaultdict(list)
for h, r, f in zip(hypotheses, references, fonts):
    font_hyp[f].append(h); font_ref[f].append(r)

print(f"\n  {'Fuente':28} {'N':>4} {'CER':>8} {'WER':>8} {'LineAcc':>9} "
      f"{'Char-F1':>9}   {'IC 95% sobre CER':>22}")
print(f"  {'─'*28} {'─'*4} {'─'*8} {'─'*8} {'─'*9} {'─'*9}   {'─'*22}")

per_font = {}
for font in sorted(font_hyp.keys()):
    hyps  = font_hyp[font]
    refs  = font_ref[font]
    n     = len(hyps)
    fm    = metrics.compute_all_metrics(hyps, refs)
    f_cer = [metrics.cer(h, r) for h, r in zip(hyps, refs)]
    ci    = metrics.bootstrap_ci(f_cer, n_bootstrap=10_000, seed=42)
    per_font[font] = {
        "N": n, "CER": fm["CER"], "WER": fm["WER"],
        "LineAcc": fm["LineAcc"], "Char_F1": fm["Char_F1"],
        "CER_std": float(np.std(f_cer, ddof=1)),
        "CER_ci_low": ci["ci_low"], "CER_ci_high": ci["ci_high"],
    }
    print(f"  {font:28} {n:4} {fm['CER']:8.4f} {fm['WER']:8.4f} {fm['LineAcc']:9.4f} "
          f"{fm['Char_F1']:9.4f}   [{ci['ci_low']:.4f}, {ci['ci_high']:.4f}]")

# Kruskal-Wallis sobre CER entre fuentes
kw_groups = {f: [metrics.cer(h, r) for h, r in zip(font_hyp[f], font_ref[f])]
             for f in font_hyp}
kw = metrics.kruskal_wallis(kw_groups)
print(f"\n  Prueba de Kruskal-Wallis sobre el CER entre las {kw['n_groups']} fuentes:")
print(f"    H = {kw['statistic']:.4f}")
print(f"    p = {kw['p_value']:.4f}")
print(f"    {'Rechaza' if kw['p_value'] < 0.05 else 'No rechaza'} la igualdad "
      f"de distribuciones entre fuentes (α = 0,05).")

# 9. Volcado final en JSON

In [ ]:
output = {
    "test_set": {
        "n_total":  len(hypotheses),
        "n_fonts":  len(font_hyp),
        "fonts":    sorted(font_hyp.keys()),
    },
    "global_greedy": {
        "CER":          m_global["CER"],
        "WER":          m_global["WER"],
        "LineAcc":      m_global["LineAcc"],
        "Char_F1":      m_global["Char_F1"],
        "BLEU4_char":   m_global["BLEU4_char"],
        "Exact_matches": m_global["Exact_matches"],
        "Avg_ref_len_char": m_global["Avg_ref_len_char"],
        "Avg_hyp_len_char": m_global["Avg_hyp_len_char"],
    },
    "global_bootstrap": {
        name: {
            "mean":    ci["mean"],
            "std":     float(np.std(
                [metrics.cer(h, r) for h, r in zip(hypotheses, references)] if name == "CER" else
                [metrics.wer(h, r) for h, r in zip(hypotheses, references)] if name == "WER" else
                [1.0 if h == r else 0.0 for h, r in zip(hypotheses, references)] if name == "Exactitud por línea" else
                [metrics._char_prf(h, r)[2] for h, r in zip(hypotheses, references)],
                ddof=1)),
            "ci_low":  ci["ci_low"],
            "ci_high": ci["ci_high"],
        }
        for name, ci in ci_summary.items()
    },
    "per_font_greedy": per_font,
    "kruskal_wallis_fonts": {
        "H":       kw["statistic"],
        "p_value": kw["p_value"],
        "n_groups": kw["n_groups"],
    },
    "model_config": {k: cfg.get(k) for k in
                     ("vocab_size", "img_height", "hidden_size",
                      "num_layers", "dropout") if k in cfg},
    "checkpoint_path": str(CHECKPOINT),
}

print("\n" + "=" * 72)
print("COPIA EL BLOQUE COMPLETO ABAJO Y PÉGALO COMO RESPUESTA")
print("=" * 72)
print()
print(json.dumps(output, indent=2, ensure_ascii=False))

# 10. (Opcional) Evaluación con beam + LM 5-grama

Solo se ejecuta si tienes el fichero `lm_5gram.arpa` montado como dataset
y el paquete `kenlm` instalado. Esta celda mide el rendimiento del decoder
de producción (beam + LM) y resulta interesante para la comparación con
la decodificación greedy reportada en cap. 4. Si no tienes el LM cargado,
salta esta celda.

In [ ]:
# Descomenta y edita las dos primeras líneas si vas a ejecutar este bloque
# LM_PATH = Path("/kaggle/input/<dataset-lm>/lm_5gram.arpa")
# LM_ALPHA = 0.4
#
# !pip -q install https://github.com/kpu/kenlm/archive/master.zip
#
# import kenlm
# from dataset import decode_ctc_beam
#
# lm = kenlm.Model(str(LM_PATH))
#
# hyp_beam = []
# print("Ejecutando beam + LM...")
# with torch.no_grad():
#     for i in tqdm(range(0, len(items), BATCH_SIZE)):
#         batch_items = items[i:i+BATCH_SIZE]
#         batch, widths, bf, br = collate_pad(batch_items)
#         batch = batch.to(device, non_blocking=True)
#         log_probs = model(batch).permute(1, 0, 2)   # [B, T, V]
#         for j, (lp, w) in enumerate(zip(log_probs, widths)):
#             valid_t = w // CNN_STRIDE
#             seq = lp[:valid_t].cpu().float().numpy().tolist()
#             hyp_beam.append(decode_ctc_beam(
#                 seq, beam_width=10, blank_bonus=2.0,
#                 length_norm_alpha=0.65, lm=lm, lm_alpha=LM_ALPHA))
#
# m_beam = metrics.compute_all_metrics(hyp_beam, references)
# print(f"\n  CER (beam+LM): {m_beam['CER']:.4f}")
# print(f"  WER (beam+LM): {m_beam['WER']:.4f}")
# print(f"  LineAcc (beam+LM): {m_beam['LineAcc']:.4f}")